# Basic Components

In [1]:
import pandas as pd,pyomo.environ as pe
m=pe.ConcreteModel()
type(m)

pyomo.core.base.PyomoModel.ConcreteModel

## Objectives

In [2]:
m.x=pe.Var(within=pe.NonNegativeReals)
m.y=pe.Var(bounds=(0,None))
m.obj=pe.Objective(expr=4*m.x+3*m.y,sense=pe.maximize)
m.obj.pprint()

obj : Size=1, Index=None, Active=True
    Key  : Active : Sense    : Expression
    None :   True : maximize : 4*x + 3*y


## Constraints

In [3]:
m.c1=pe.Constraint(expr=m.x<=4)
m.c2=pe.Constraint(expr=2*m.x+m.y<=10)
m.c3=pe.Constraint(expr=(None,m.x+m.y,10))
m.pprint()

2 Var Declarations
    x : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True : NonNegativeReals
    y : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True :  Reals

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 4*x + 3*y

3 Constraint Declarations
    c1 : Size=1, Index=None, Active=True
        Key  : Lower : Body : Upper : Active
        None :  -Inf :    x :   4.0 :   True
    c2 : Size=1, Index=None, Active=True
        Key  : Lower : Body    : Upper : Active
        None :  -Inf : 2*x + y :  10.0 :   True
    c3 : Size=1, Index=None, Active=True
        Key  : Lower : Body  : Upper : Active
        None :  -Inf : x + y :  10.0 :   True

6 Declarations: x y obj c1 c2 c3


# Solving Optimisation Problems
## Calling Solvers From Pyomo

In [4]:
opt=pe.SolverFactory('appsi_highs')
log=opt.solve(m,tee=True)
log.write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 30.0
  Upper bound: 30.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: maximize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


In [5]:
assert log.solver.status==pe.SolverStatus.ok
pe.SolverStatus._member_names_

['ok', 'warning', 'error', 'aborted', 'unknown']

# Retrieving Optimisation Results
## Primal Values

In [6]:
m.obj()

30.0

In [7]:
m.x()

-0.0

In [8]:
m.y()

10.0

## Dual Values

In [9]:
m.dual=pe.Suffix(direction=pe.Suffix.IMPORT)
pe.SolverFactory('appsi_highs').solve(m).write()
m.dual[m.c1]

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 30.0
  Upper bound: 30.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: maximize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


0.0

In [10]:
m.dual[m.c2]

1.0

# Indexed Components, Sets
## Range

In [11]:
m=pe.ConcreteModel()
m.A=pe.RangeSet(2)
m.A.pprint()

A : Dimen=1, Size=2, Bounds=(1, 2)
    Key  : Finite : Members
    None :   True :   [1:2]


In [12]:
B=['wind','solar']
m.B=pe.Set(initialize=B)
m.B.pprint()

B : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :    2 : {'wind', 'solar'}


## Indexed Variables

In [13]:
m.x=pe.Var(m.A)
m.x.pprint()

x : Size=2, Index=A
    Key : Lower : Value : Upper : Fixed : Stale : Domain
      1 :  None :  None :  None : False :  True :  Reals
      2 :  None :  None :  None : False :  True :  Reals


In [14]:
m.y=pe.Var(m.A,
           m.B)
m.y.pprint()

y : Size=4, Index=A*B
    Key          : Lower : Value : Upper : Fixed : Stale : Domain
    (1, 'solar') :  None :  None :  None : False :  True :  Reals
     (1, 'wind') :  None :  None :  None : False :  True :  Reals
    (2, 'solar') :  None :  None :  None : False :  True :  Reals
     (2, 'wind') :  None :  None :  None : False :  True :  Reals


## List Comprehension On Indexed Variables

In [15]:
m.z=pe.Var(B)
m.c1=pe.Constraint(expr=sum(m.z[b] for b in m.B)<=10)
m.c1.pprint()

c1 : Size=1, Index=None, Active=True
    Key  : Lower : Body               : Upper : Active
    None :  -Inf : z[wind] + z[solar] :  10.0 :   True


## Indexed Constraints

In [16]:
def construction_rule(m,a):
    return sum(m.y[a,b] for b in m.B)==1
m.c2=pe.Constraint(m.A,rule=construction_rule)
m.c2.pprint()

c2 : Size=2, Index=A, Active=True
    Key : Lower : Body                   : Upper : Active
      1 :   1.0 : y[1,wind] + y[1,solar] :   1.0 :   True
      2 :   1.0 : y[2,wind] + y[2,solar] :   1.0 :   True


In [17]:
m.c2[2].pprint()

{Member of c2} : Size=2, Index=A, Active=True
    Key : Lower : Body                   : Upper : Active
      2 :   1.0 : y[2,wind] + y[2,solar] :   1.0 :   True


# Electricity Market Examples
## Single Bidding Zone, Period


In [18]:
m=pe.ConcreteModel()
m.dual=pe.Suffix(direction=pe.Suffix.IMPORT)
capacities={'Coal':35000,
            'Wind':3000,
            'Gas':8000,
            'Oil':2000}
m.S=pe.Set(initialize=capacities.keys())
m.g=pe.Var(m.S,within=pe.NonNegativeReals)
marginal_costs={'Wind':0,
                'Coal':30,
                'Gas':60,
                'Oil':80}
m.cost=pe.Objective(expr=sum(marginal_costs[s]*m.g[s] for s in m.S))
m.cost.pprint()

cost : Size=1, Index=None, Active=True
    Key  : Active : Sense    : Expression
    None :   True : minimize : 30*g[Coal] + 0*g[Wind] + 60*g[Gas] + 80*g[Oil]


In [19]:
@m.Constraint(m.S)
def generator_limit(m,s):
    return m.g[s]<=capacities[s]
m.generator_limit.pprint()

generator_limit : Size=4, Index=S, Active=True
    Key  : Lower : Body    : Upper   : Active
    Coal :  -Inf : g[Coal] : 35000.0 :   True
     Gas :  -Inf :  g[Gas] :  8000.0 :   True
     Oil :  -Inf :  g[Oil] :  2000.0 :   True
    Wind :  -Inf : g[Wind] :  3000.0 :   True


In [20]:
load=42000
m.energy_balance=pe.Constraint(expr=sum(m.g[s] for s in m.S)==load)
m.energy_balance.pprint()

energy_balance : Size=1, Index=None, Active=True
    Key  : Lower   : Body                                : Upper   : Active
    None : 42000.0 : g[Coal] + g[Wind] + g[Gas] + g[Oil] : 42000.0 :   True


In [21]:
pe.SolverFactory('appsi_highs').solve(m).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 1290000.0
  Upper bound: 1290000.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


In [22]:
pd.Series(m.g.get_values())

Coal    35000.0
Wind     3000.0
Gas      4000.0
Oil         0.0
dtype: float64

In [23]:
market_price=m.dual[m.energy_balance]
market_price

60.0

In [24]:
pd.Series(m.dual.values(),
          m.dual.keys())

generator_limit[Coal]   -30.0
generator_limit[Wind]   -60.0
generator_limit[Gas]     -0.0
generator_limit[Oil]     -0.0
[None]                   60.0
dtype: float64

In [25]:
pd.Series({s:m.dual[m.generator_limit[s]] for s in m.S})

Coal   -30.0
Wind   -60.0
Gas     -0.0
Oil     -0.0
dtype: float64

## 2 Bidding Zones With Transmission

In [26]:
loads={'South Africa':42000,
       'Mozambique':650}
capacities={'South Africa':{'Coal':35000,
                            'Wind':3000,
                            'Gas':8000,
                            'Oil':2000},
            'Mozambique':{'Hydro':1200}}
marginal_costs['Hydro']=0
transmission=500
m=pe.ConcreteModel()
m.dual=pe.Suffix(direction=pe.Suffix.IMPORT)
m.countries=pe.Set(initialize=loads.keys())
technologies=list(capacities['South Africa'].keys()|capacities['Mozambique'])
m.technologies=pe.Set(initialize=technologies)
m.g=pe.Var(m.countries,m.technologies,within=pe.NonNegativeReals)
m.f=pe.Var()
m.cost=pe.Objective(expr=sum(marginal_costs[s]*m.g[c,s] for s in m.technologies for c in m.countries))
@m.Constraint(m.countries,
              m.technologies)
def generator_limit(m,c,s):
    return m.g[c,s]<=capacities[c].get(s,0)
m.generator_limit.pprint()

generator_limit : Size=10, Index=countries*technologies, Active=True
    Key                       : Lower : Body                  : Upper   : Active
       ('Mozambique', 'Coal') :  -Inf :    g[Mozambique,Coal] :     0.0 :   True
        ('Mozambique', 'Gas') :  -Inf :     g[Mozambique,Gas] :     0.0 :   True
      ('Mozambique', 'Hydro') :  -Inf :   g[Mozambique,Hydro] :  1200.0 :   True
        ('Mozambique', 'Oil') :  -Inf :     g[Mozambique,Oil] :     0.0 :   True
       ('Mozambique', 'Wind') :  -Inf :    g[Mozambique,Wind] :     0.0 :   True
     ('South Africa', 'Coal') :  -Inf :  g[South Africa,Coal] : 35000.0 :   True
      ('South Africa', 'Gas') :  -Inf :   g[South Africa,Gas] :  8000.0 :   True
    ('South Africa', 'Hydro') :  -Inf : g[South Africa,Hydro] :     0.0 :   True
      ('South Africa', 'Oil') :  -Inf :   g[South Africa,Oil] :  2000.0 :   True
     ('South Africa', 'Wind') :  -Inf :  g[South Africa,Wind] :  3000.0 :   True


In [27]:
@m.Constraint(m.countries)
def kcl(m,c):
    sign=-1 if c=='Mozambique' else 1
    return sum(m.g[c,s] for s in m.technologies)-sign*m.f==loads[c]
m.kcl.pprint()

kcl : Size=2, Index=countries, Active=True
    Key          : Lower   : Body                                                                                                                : Upper   : Active
      Mozambique :   650.0 :           g[Mozambique,Coal] + g[Mozambique,Oil] + g[Mozambique,Hydro] + g[Mozambique,Gas] + g[Mozambique,Wind] + f :   650.0 :   True
    South Africa : 42000.0 : g[South Africa,Coal] + g[South Africa,Oil] + g[South Africa,Hydro] + g[South Africa,Gas] + g[South Africa,Wind] - f : 42000.0 :   True


In [28]:
m.line_limit=pe.Constraint(expr=(-transmission,m.f,transmission))
m.line_limit.pprint()

line_limit : Size=1, Index=None, Active=True
    Key  : Lower  : Body : Upper : Active
    None : -500.0 :    f : 500.0 :   True


In [29]:
pe.SolverFactory('appsi_highs').solve(m).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 1260000.0
  Upper bound: 1260000.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


In [30]:
m.cost()

1260000.0

In [31]:
pd.Series(m.g.get_values()).unstack()

,Coal,Gas,Hydro,Oil,Wind
Mozambique,0.0,0.0,1150.0,0.0,0.0
South Africa,35000.0,3500.0,-0.0,0.0,3000.0


In [32]:
m.f()

-500.0

In [33]:
pd.Series(m.dual.values(),
          m.dual.keys())

generator_limit[South Africa,Coal]    -30.0
generator_limit[South Africa,Oil]      -0.0
generator_limit[South Africa,Hydro]   -60.0
generator_limit[South Africa,Gas]      -0.0
generator_limit[South Africa,Wind]    -60.0
generator_limit[Mozambique,Coal]       -0.0
generator_limit[Mozambique,Oil]        -0.0
generator_limit[Mozambique,Hydro]      -0.0
generator_limit[Mozambique,Gas]        -0.0
generator_limit[Mozambique,Wind]       -0.0
kcl[South Africa]                      60.0
kcl[Mozambique]                        -0.0
[None]                                 60.0
dtype: float64

In [34]:
m.line_limit.deactivate()
m.g['South Africa','Coal'].fix(34000)
pe.SolverFactory('appsi_highs').solve(m).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 1287000.0
  Upper bound: 1287000.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


In [35]:
pd.Series(m.g.get_values()).unstack()

,Coal,Gas,Hydro,Oil,Wind
Mozambique,-0.0,0.0,1200.0,0.0,-0.0
South Africa,34000.0,4450.0,-0.0,0.0,3000.0


In [36]:
m.cost()

1287000.0

## Single Bidding Zone With Several Periods

In [37]:
wind_ts=[.3,.6,.4,.5]
load_ts=[42000,43000,45000,46000]
m=pe.ConcreteModel()
m.dual=pe.Suffix(direction=pe.Suffix.IMPORT)
techs=capacities['South Africa'].keys()
m.S=pe.Set(initialize=techs)
m.T=pe.RangeSet(4)
m.g=pe.Var(m.S,m.T,within=pe.NonNegativeReals)
m.cost=pe.Objective(expr=sum(marginal_costs[s]*m.g[s,t] for s in m.S for t in m.T))
@m.Constraint(m.S,
              m.T)
def generator_limit(m,s,t):
    cf=wind_ts[t-1] if s=='Wind' else 1
    return m.g[s,t]<=cf*capacities['South Africa'][s]
m.generator_limit.pprint()

generator_limit : Size=16, Index=S*T, Active=True
    Key         : Lower : Body      : Upper   : Active
    ('Coal', 1) :  -Inf : g[Coal,1] : 35000.0 :   True
    ('Coal', 2) :  -Inf : g[Coal,2] : 35000.0 :   True
    ('Coal', 3) :  -Inf : g[Coal,3] : 35000.0 :   True
    ('Coal', 4) :  -Inf : g[Coal,4] : 35000.0 :   True
     ('Gas', 1) :  -Inf :  g[Gas,1] :  8000.0 :   True
     ('Gas', 2) :  -Inf :  g[Gas,2] :  8000.0 :   True
     ('Gas', 3) :  -Inf :  g[Gas,3] :  8000.0 :   True
     ('Gas', 4) :  -Inf :  g[Gas,4] :  8000.0 :   True
     ('Oil', 1) :  -Inf :  g[Oil,1] :  2000.0 :   True
     ('Oil', 2) :  -Inf :  g[Oil,2] :  2000.0 :   True
     ('Oil', 3) :  -Inf :  g[Oil,3] :  2000.0 :   True
     ('Oil', 4) :  -Inf :  g[Oil,4] :  2000.0 :   True
    ('Wind', 1) :  -Inf : g[Wind,1] :   900.0 :   True
    ('Wind', 2) :  -Inf : g[Wind,2] :  1800.0 :   True
    ('Wind', 3) :  -Inf : g[Wind,3] :  1200.0 :   True
    ('Wind', 4) :  -Inf : g[Wind,4] :  1500.0 :   True


In [38]:
@m.Constraint(m.T)
def energy_balance(m,t):
    return sum(m.g[s,t] for s in m.S)==load_ts[t-1]
m.energy_balance.pprint()

energy_balance : Size=4, Index=T, Active=True
    Key : Lower   : Body                                        : Upper   : Active
      1 : 42000.0 : g[Coal,1] + g[Wind,1] + g[Gas,1] + g[Oil,1] : 42000.0 :   True
      2 : 43000.0 : g[Coal,2] + g[Wind,2] + g[Gas,2] + g[Oil,2] : 43000.0 :   True
      3 : 45000.0 : g[Coal,3] + g[Wind,3] + g[Gas,3] + g[Oil,3] : 45000.0 :   True
      4 : 46000.0 : g[Coal,4] + g[Wind,4] + g[Gas,4] + g[Oil,4] : 46000.0 :   True


In [39]:
%timeit pe.SolverFactory('appsi_highs').solve(m).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 6082000.0
  Upper bound: 6082000.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0
# ===========================================

In [40]:
m.cost()

6082000.0

In [41]:
pd.Series(m.dual.values(),
          m.dual.keys())

generator_limit[Coal,1]   -30.0
generator_limit[Coal,2]   -30.0
generator_limit[Coal,3]   -50.0
generator_limit[Coal,4]   -50.0
generator_limit[Wind,1]   -60.0
generator_limit[Wind,2]   -60.0
generator_limit[Wind,3]   -80.0
generator_limit[Wind,4]   -80.0
generator_limit[Gas,1]     -0.0
generator_limit[Gas,2]     -0.0
generator_limit[Gas,3]    -20.0
generator_limit[Gas,4]    -20.0
generator_limit[Oil,1]     -0.0
generator_limit[Oil,2]     -0.0
generator_limit[Oil,3]     -0.0
generator_limit[Oil,4]     -0.0
energy_balance[1]          60.0
energy_balance[2]          60.0
energy_balance[3]          80.0
energy_balance[4]          80.0
dtype: float64

In [42]:
pd.Series(m.g.get_values()).unstack(0)

,Coal,Gas,Oil,Wind
1,35000.0,6100.0,0.0,900.0
2,35000.0,6200.0,0.0,1800.0
3,35000.0,8000.0,800.0,1200.0
4,35000.0,8000.0,1500.0,1500.0


## Single Bidding Zone With Several Periods, Storage

In [43]:
storage_energy=6000
storage_power=1000
efficiency=.9
standing_loss=.00001
m.discharge=pe.Var(m.T,bounds=(0,storage_power))
m.charge=pe.Var(m.T,bounds=(0,storage_power))
m.soc=pe.Var(m.T,bounds=(0,storage_energy))
@m.Constraint(m.T)
def storage_consistency(m,t):
    if t==1:
        return m.soc[t]==0
    return (m.soc[t]==(1-standing_loss)*m.soc[t-1]+efficiency*m.charge[t]-1/efficiency*m.discharge[t])
m.storage_consistency.pprint()

storage_consistency : Size=4, Index=T, Active=True
    Key : Lower : Body                                                                        : Upper : Active
      1 :   0.0 :                                                                      soc[1] :   0.0 :   True
      2 :   0.0 : soc[2] - (0.99999*soc[1] + 0.9*charge[2] - 1.1111111111111112*discharge[2]) :   0.0 :   True
      3 :   0.0 : soc[3] - (0.99999*soc[2] + 0.9*charge[3] - 1.1111111111111112*discharge[3]) :   0.0 :   True
      4 :   0.0 : soc[4] - (0.99999*soc[3] + 0.9*charge[4] - 1.1111111111111112*discharge[4]) :   0.0 :   True


In [44]:
m.del_component(m.energy_balance)
@m.Constraint(m.T)
def energy_balance(m,t):
    return sum(m.g[s,t] for s in m.S)+m.discharge[t]-m.charge[t]==load_ts[t-1]
pe.SolverFactory('appsi_highs').solve(m).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 6017200.65599352
  Upper bound: 6017200.65599352
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


In [45]:
m.cost()

6017200.65599352

In [46]:
pd.Series(m.g.get_values()).unstack(0)

,Coal,Gas,Oil,Wind
1,35000.0,5100.0,0.0000,900.0
2,35000.0,7200.0,0.0000,1800.0
3,35000.0,8000.0,0.0000,1200.0
4,35000.0,8000.0,1490.0082,1500.0


In [47]:
pd.Series(m.charge.get_values())

1       0.0
2    1000.0
3       0.0
4       0.0
dtype: float64

In [48]:
pd.Series(m.discharge.get_values())

1    1000.0000
2       0.0000
3     800.0000
4       9.9918
dtype: float64

In [49]:
pd.Series(m.soc.get_values())

1     -0.000000
2    900.000000
3     11.102111
4      0.000000
dtype: float64

In [50]:
pd.Series(m.dual.values(),
          m.dual.keys())

generator_limit[Coal,1]       -30.00000
generator_limit[Coal,2]       -30.00000
generator_limit[Coal,3]       -49.99920
generator_limit[Coal,4]       -50.00000
generator_limit[Wind,1]       -60.00000
generator_limit[Wind,2]       -60.00000
generator_limit[Wind,3]       -79.99920
generator_limit[Wind,4]       -80.00000
generator_limit[Gas,1]         -0.00000
generator_limit[Gas,2]         -0.00000
generator_limit[Gas,3]        -19.99920
generator_limit[Gas,4]        -20.00000
generator_limit[Oil,1]         -0.00000
generator_limit[Oil,2]         -0.00000
generator_limit[Oil,3]         -0.00000
generator_limit[Oil,4]         -0.00000
[Unattached ConstraintData]    60.00000
[Unattached ConstraintData]    60.00000
[Unattached ConstraintData]    80.00000
[Unattached ConstraintData]    80.00000
storage_consistency[1]        -71.99784
storage_consistency[2]        -71.99856
storage_consistency[3]        -71.99928
storage_consistency[4]        -72.00000
energy_balance[1]              60.00000
